# **Recurrent Language Modeling**
<img src="https://static01.nyt.com/images/2018/05/15/arts/01hal-voice1/merlin_135847308_098289a6-90ee-461b-88e2-20920469f96a-superJumbo.jpg" width=20% style="border-radius:20px"><br>
Let's teach computer to speak... (cause what can go wrong?)

## Table Of contents
- What is this notebook about?
- What are RNNs?
- Character-level RNN LM Implementation
- What are LSTMs and GRUs?
- Character-level LSTM and GRU LMs Implementation
- Language Modeling fun (cause why not?)
- Word Level language Modeling
- Word embeddings
- Word-level RNN LM Implementation
- Word-level LSTM and GRU LMs Implementation
- More fun!
- **Tiny Stories** and Finita La Comedia!

### What is this notebook about?
This is an introductory-level guide to language modeling with RNNs and LSTMs. and GRUs!<br>
In this one my friends we will not only discuss all the details of Recurrent Language Modeling but we will also apply it to interesting tasks, such as:
- Novel writing
- Programming (oh no... AGI takes my job!)
- LaTex
- Story Extending (Balabebe)

**Sounds Interesting? Good!**

### What are RNNs?
RNN - Recurrent Neural Network - special framework (type of neural networks), which processes input sequentially sharing all the parameters.<br>
It applies the same function to each sequence token and carries information about the past in special **hidden state**.<br>
There are various RNN visualizations, but find all of the quite confusing, because they either express nothing or give wrong ideas about practical implementation.<br>
On the image below you see a visualization of RNN wrapped out in time.<br>
<img src="https://i.imgur.com/Iveichs.jpg" style="border-radius: 20px" width=30%><br>
As you can see on each timestep RNN (green block) processes an input (character in this case) and a hidden state (it comes from left to right through the time).<br>
On each time step RNN returns an output token (at timestep t=1 it recieves input "h" and returns output "e". On the scheme visually it goes up) and updates hidden state (this one goes to the right).<br>
On intuitive level it learns to predict the next token given input token and using the previous knowledge stored in hidden state.<br>
**Important technical note!**
RNN is not a set of individual blocks, it is a one Module, which is called multiple times accumulating gradients.<br>
In other words, on practice it's more like this:<br>
<img src="https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/RNN-rolled.png" width=10%><br>
But this scheme explaines nothing at all.<br>

**Important sanity note!**
RNNs are not only used for Language Modeling. They can play a role of an encoder for classification/regression tasks, they are used to work with time-series data. They are capable of more things!

But what about backpropagation? How is it possible?<br>
Well, it's called **"Backpropagation through time"** and for a good reason.<br>
At each timestep we calculate loss. We take average loss over timesteps and backpropagate from it.<br>
<img src="https://wikidocs.net/images/page/160068/7_Backpropagation-in-RNNs.jpg" style="border-radius:20px" width=30%><br>

### Character-level RNN LM Implementation

In [191]:
import torch
from torch import nn, optim
import torch.nn.functional as F
from tqdm import tqdm

In [192]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [193]:
with open("../../Data/NLP/onegin.txt", "r") as f:
    data = f.read()

In [194]:
print(data[:80])


Не мысля гордый свет забавить,
Вниманье дружбы возлюбя,
Хотел бы я тебе предста


In [195]:
# vocabulary of unique symbols
vocab = sorted(set(data)) + ["<", ">"]
vocab_size = len(vocab)

# character-index encoding and vice-a-versa
itos = {i+2:s for i, s in enumerate(vocab)}
itos[0] = "<"
itos[1] = ">"
stoi = {s:i for i, s in itos.items()}

print(vocab_size)

147


In [196]:
stoi["a"], itos[41]

(48, 'R')

In [197]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        """
        Why does input_to hidden has input size + hidden size?
        Because it recieves hidden size from the previous time step.
        Initially it is set to zeros.
        In math notation we have separate parameter matrices for input and hidden
        But on practice it's more convenient to concatenate them
        And use a shared parameter matrix
        """
        self.input_to_hidden = nn.Linear(input_size + hidden_size, hidden_size)
        self.activation = nn.Tanh()
        self.hidden_to_output = nn.Linear(hidden_size, output_size)

    def init_hidden(self, device):
        return torch.zeros(1, self.hidden_size, device=device)
    
    def forward(self, input, hidden):
        input_with_hidden = torch.cat([input, hidden], dim=1)
        new_hidden_state = self.activation(
            self.input_to_hidden(input_with_hidden)
        )
        output = self.hidden_to_output(new_hidden_state)
        return output, new_hidden_state

Some things may not make sense at this moment, but it's okay.<br>
Once you see the training loop it should be more meaningful and intuitive.

In [198]:
from nltk import sent_tokenize  # we will process sentences, not the entire text

In [199]:
">" in vocab or "<" in vocab

True

In [200]:
sentences = sent_tokenize(data)
# add a "start of sequence" token and "end of sequence" token
sentences = ["<" + s + ">" for s in sentences]
sentences

['<\nНе мысля гордый свет забавить,\nВниманье дружбы возлюбя,\nХотел бы я тебе представить\nЗалог достойнее тебя,\nДостойнее души прекрасной,\nСвятой исполненной мечты,\nПоэзии живой и ясной,\nВысоких дум и простоты;\nНо так и быть — рукой пристрастной\nПрими собранье пестрых глав,\nПолусмешных, полупечальных,\nПростонародных, идеальных,\nНебрежный плод моих забав,\nБессонниц, легких вдохновений,\nНезрелых и увядших лет,\nУма холодных наблюдений\nИ сердца горестных замет.>',
 '<9\nГЛАВА ПЕРВАЯ\n\nИ жить торопится и чувствовать спешит.>',
 '<Кн.>',
 '<Вяземский.>',
 '<I\n\n«Мой дядя самых честных правил,\nКогда не в шутку занемог,\nОн уважать себя заставил\nИ лучше выдумать не мог.>',
 '<Его пример другим наука;\nНо, боже мой, какая скука\nС больным сидеть и день и ночь,\nНе отходя ни шагу прочь!>',
 '<Какое низкое коварство\nПолуживого забавлять,\nЕму подушки поправлять,\nПечально подносить лекарство,\nВздыхать и думать про себя:\nКогда же черт возьмет тебя!»\nII\n\nТак думал молодой п

In [201]:
def generate(model):
    inp = "<"
    hid = model.init_hidden(device)
    with torch.inference_mode():
        while True:
            input = torch.tensor([stoi[inp[-1]]])
            input = F.one_hot(input, vocab_size).to(device)
            out, hid = model(input, hid)
            out = torch.multinomial(torch.softmax(out, dim=1), 1).item()
            if itos[out] == ">": break
            inp += itos[out]
    return inp[1:]

In [202]:
def train_function(model, criterion, optimizer, epochs=10):
    for epoch in range(epochs):
        model.train()
        for sentence in tqdm(sentences):
            input = sentence[:][:-1]
            target = sentence[:][1:]
            input = torch.tensor([stoi[ch] for ch in input]).to(device)
            target = torch.tensor([stoi[ch] for ch in target]).to(device)
            input_ohe = F.one_hot(input, vocab_size).to(device)
            
            loss = 0
            hidden = model.init_hidden(device)
            for input_char, target_char in zip(input_ohe, target):
                input_char = input_char.unsqueeze(0)
                target_char = target_char.unsqueeze(0)
                out, hidden = model(input_char, hidden)
                loss_t = criterion(out, target_char)
                loss += loss_t
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        print(f"Epoch: {epoch+1} | {loss.item()}")
        print(generate(model))
    
    return model

In [204]:
sentences = [s for s in sentences if s != "<.>"]

In [205]:
len(sentences)

1700

In [206]:
sentences[:5]

['<\nНе мысля гордый свет забавить,\nВниманье дружбы возлюбя,\nХотел бы я тебе представить\nЗалог достойнее тебя,\nДостойнее души прекрасной,\nСвятой исполненной мечты,\nПоэзии живой и ясной,\nВысоких дум и простоты;\nНо так и быть — рукой пристрастной\nПрими собранье пестрых глав,\nПолусмешных, полупечальных,\nПростонародных, идеальных,\nНебрежный плод моих забав,\nБессонниц, легких вдохновений,\nНезрелых и увядших лет,\nУма холодных наблюдений\nИ сердца горестных замет.>',
 '<9\nГЛАВА ПЕРВАЯ\n\nИ жить торопится и чувствовать спешит.>',
 '<Кн.>',
 '<Вяземский.>',
 '<I\n\n«Мой дядя самых честных правил,\nКогда не в шутку занемог,\nОн уважать себя заставил\nИ лучше выдумать не мог.>']

In [207]:
model = CharRNN(vocab_size, 256, vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

model = train_function(model, criterion, optimizer)

  0%|          | 0/1700 [00:00<?, ?it/s]

100%|██████████| 1700/1700 [00:55<00:00, 30.82it/s]


Epoch: 1 | 170.87928771972656
Миесомовшете; бет я жсре пуседа демллясь, р
Прувит толденай бые косвад.


100%|██████████| 1700/1700 [00:55<00:00, 30.81it/s]


Epoch: 2 | 167.8295440673828
II4
На:
Наверноля с кушной.


...

Epoch: 8 | 153.78076171875
Ит намиша.


100%|██████████| 1700/1700 [00:54<00:00, 31.20it/s]


Epoch: 9 | 150.35992431640625
Свое ТиРусалый, в ней нене
Ваторовы мезвра жив,
И тан земел:
Азминов ненян молнакоза
Стири взагол, как пострат.


100%|██████████| 1700/1700 [00:55<00:00, 30.65it/s]

Epoch: 10 | 146.4678955078125
Мой  зомы зиканал.


In [210]:
for _ in range(5):
    print(generate(model))

Как и писардцвою вдруг,
Быенец.
Доум, так Мурт в постелы нав жерица,
И буз лестаром
Пура на бидетного польма.
Коезанись с нив емой
Ил О пере роскичей екейси озволал.
Неров, ча-тек ледвуника;
Меса глазиье свет поколсевать
И и с каком опоснеда,
Томе своданный изола;
Сворум одруж,
Я плеконы затстовя молодается розолиби с новостито полых
а на сердю.
XLII

А чит клода с Опреба дверерая.
